# 1. Introduction

CNN based models for galaxy classification.

# 2. Dataset 

The objective is to classify the given galaxy images.


The dataset consists of 17,736 images, each being a 256x256 RGB matrix, categorized into 10 classes.

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import os
import copy
import math
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

- **`Galaxy10_DECals.h5`** — your training data  

In [ ]:
# ── Load the training data ───────────────────────────────────
import h5py

train_h5_candidates = [
    './Galaxy10_DECals.h5',
    'Galaxy10_DECals.h5',
    '/home/yangz2/code/space_universe/data/A5/Galaxy10_DECals.h5',
]

train_h5 = None
for cand in train_h5_candidates:
    if os.path.exists(cand):
        train_h5 = cand
        break

if train_h5 is None:
    raise FileNotFoundError('Galaxy10_DECals.h5 not found. Please place it in the current working directory.')

with h5py.File(train_h5, 'r') as f:
    images_all = f['images'][:]   # (N, 256, 256, 3)
    labels_all = f['ans'][:]      # (N,)

print(f'Loaded training pool from: {train_h5}')
print(f'Images shape: {images_all.shape}')
print(f'Labels shape: {labels_all.shape}')

In [ ]:
# ── Stratified split: local train / local validation ─────────
# The official held-out test set is Galaxy10_DECals_20pct.h5 and is NOT used here.
X_train, X_val, y_train, y_val = train_test_split(
    images_all, labels_all,
    test_size=0.20,
    random_state=42,
    stratify=labels_all
)

print(f'Train set : {X_train.shape}')
print(f'Val set   : {X_val.shape}')

train_counts = np.bincount(y_train, minlength=10)
val_counts   = np.bincount(y_val, minlength=10)

print('\nPer-class counts (train):', train_counts)
print('Per-class counts (val)  :', val_counts)

In [ ]:
galaxy_classes = [
    'Disturbed Galaxies',
    'Merging Galaxies',
    'Round Smooth Galaxies',
    'In-between Round Smooth Galaxies',
    'Cigar Shaped Smooth Galaxies',
    'Barred Spiral Galaxies',
    'Unbarred Tight Spiral Galaxies',
    'Unbarred Loose Spiral Galaxies',
    'Edge-on Galaxies without Bulge',
    'Edge-on Galaxies with Bulge',
]

# Galaxy10 dataset (17736 images)
# ├── Class 0 (1081 images): Disturbed Galaxies
# ├── Class 1 (1853 images): Merging Galaxies
# ├── Class 2 (2645 images): Round Smooth Galaxies
# ├── Class 3 (2027 images): In-between Round Smooth Galaxies
# ├── Class 4 ( 334 images): Cigar Shaped Smooth Galaxies
# ├── Class 5 (2043 images): Barred Spiral Galaxies
# ├── Class 6 (1829 images): Unbarred Tight Spiral Galaxies
# ├── Class 7 (2628 images): Unbarred Loose Spiral Galaxies
# ├── Class 8 (1423 images): Edge-on Galaxies without Bulge
# └── Class 9 (1873 images): Edge-on Galaxies with Bulge

print('Unique classes in training set:', np.unique(y_train))

In [ ]:
# Visualise sample training images

import random

fig = plt.figure(figsize=(20, 20))
for i in range(25):
    idx = i + random.randint(0, 100)
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train[idx])
    plt.title(galaxy_classes[y_train[idx]], fontsize=8)
    fig.tight_layout(pad=3.0)
plt.show()

In [ ]:
# ── PyTorch Dataset wrapper ──────────────────────────────────

class GalaxyDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        """
        images: NumPy array of shape (N, 256, 256, 3), dtype uint8
        labels: NumPy array of shape (N,), or None for inference
        transform: torchvision transforms pipeline
        """
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]  # HWC uint8
        if self.transform is not None:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        if self.labels is None:
            return image
        return image, int(self.labels[idx])

# 3. Methodology

## 3.1 Appetizer

### 3.1.1 Simple CNN

We use a 3-layer CNN network, with batch normalisation layers.

In [ ]:
# ── Define the improved model source as a string ─────────────
model_source = """
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import convnext_small, ConvNeXt_Small_Weights

class MorphologyExtractor(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def _gini(self, gray):
        x = gray.flatten(1)
        x = torch.sort(x, dim=1)[0]
        n = x.size(1)
        idx = torch.arange(1, n + 1, device=x.device, dtype=x.dtype).view(1, -1)
        denom = x.sum(dim=1, keepdim=True).clamp_min(self.eps)
        g = (2.0 * (idx * x).sum(dim=1, keepdim=True) / (n * denom)) - (n + 1.0) / n
        return g

    def _m20(self, flux, xx, yy, xc, yc):
        dx = xx - xc
        dy = yy - yc
        m_i = flux * (dx * dx + dy * dy)
        total_m = m_i.flatten(1).sum(dim=1, keepdim=True).clamp_min(self.eps)

        flat_flux = flux.flatten(1)
        flat_m = m_i.flatten(1)
        sorted_flux, order = torch.sort(flat_flux, dim=1, descending=True)
        sorted_m = torch.gather(flat_m, 1, order)
        csum = sorted_flux.cumsum(dim=1)
        threshold = 0.2 * flat_flux.sum(dim=1, keepdim=True)
        mask = (csum <= threshold).to(sorted_m.dtype)
        top_m = (sorted_m * mask).sum(dim=1, keepdim=True).clamp_min(self.eps)
        return torch.log10((top_m / total_m).clamp_min(self.eps))

    def _sersic_proxy(self, flux, r_norm):
        # Fit log(I) ≈ a + b * r^(1/n) over a small candidate set of n values.
        candidates = [0.5, 1.0, 2.0, 3.0, 4.0, 6.0]
        mask = flux > (0.05 * flux.amax(dim=(1, 2), keepdim=True))
        y = torch.log(flux.clamp_min(self.eps))

        best_err = None
        best_n = None
        best_slope = None

        for n_val in candidates:
            x = r_norm.clamp_min(self.eps).pow(1.0 / n_val)
            w = mask.to(flux.dtype)
            sw = w.sum(dim=(1, 2)).clamp_min(10.0)
            sx = (w * x).sum(dim=(1, 2))
            sy = (w * y).sum(dim=(1, 2))
            sxx = (w * x * x).sum(dim=(1, 2))
            sxy = (w * x * y).sum(dim=(1, 2))
            denom = (sw * sxx - sx * sx).clamp_min(self.eps)
            b = (sw * sxy - sx * sy) / denom
            a = (sy - b * sx) / sw
            pred = a[:, None, None] + b[:, None, None] * x
            err = (w * (pred - y).pow(2)).sum(dim=(1, 2)) / sw

            if best_err is None:
                best_err = err
                best_n = torch.full_like(err, float(n_val))
                best_slope = b
            else:
                better = err < best_err
                best_err = torch.where(better, err, best_err)
                best_n = torch.where(better, torch.full_like(err, float(n_val)), best_n)
                best_slope = torch.where(better, b, best_slope)

        return best_n.unsqueeze(1), best_slope.unsqueeze(1), best_err.unsqueeze(1)

    def forward(self, x):
        x = x.float()
        if x.max() > 1.5:
            x = x / 255.0

        gray = 0.2989 * x[:, 0] + 0.5870 * x[:, 1] + 0.1140 * x[:, 2]
        gray = gray.clamp_min(0.0)
        B, H, W = gray.shape

        yy = torch.linspace(-1.0, 1.0, H, device=x.device, dtype=x.dtype).view(1, H, 1).expand(B, H, W)
        xx = torch.linspace(-1.0, 1.0, W, device=x.device, dtype=x.dtype).view(1, 1, W).expand(B, H, W)

        total = gray.sum(dim=(1, 2), keepdim=True).clamp_min(self.eps)
        mean_int = gray.mean(dim=(1, 2), keepdim=True)
        std_int = gray.std(dim=(1, 2), keepdim=True)

        xc = (gray * xx).sum(dim=(1, 2), keepdim=True) / total
        yc = (gray * yy).sum(dim=(1, 2), keepdim=True) / total
        dx = xx - xc
        dy = yy - yc
        rr = torch.sqrt(dx * dx + dy * dy)
        rr_max = rr.amax(dim=(1, 2), keepdim=True).clamp_min(self.eps)
        r_norm = rr / rr_max

        central = (gray * (r_norm <= 0.15)).sum(dim=(1, 2), keepdim=True) / total
        inner = (gray * (r_norm <= 0.30)).sum(dim=(1, 2), keepdim=True) / total
        outer = (gray * (r_norm >= 0.60)).sum(dim=(1, 2), keepdim=True) / total

        rot180 = torch.rot90(gray, 2, dims=(1, 2))
        asym = (gray - rot180).abs().sum(dim=(1, 2), keepdim=True) / (gray.abs().sum(dim=(1, 2), keepdim=True) + self.eps)

        smooth = F.avg_pool2d(gray.unsqueeze(1), kernel_size=7, stride=1, padding=3).squeeze(1)
        clump = (gray - smooth).abs().sum(dim=(1, 2), keepdim=True) / (gray.abs().sum(dim=(1, 2), keepdim=True) + self.eps)

        mu_xx = (gray * dx * dx).sum(dim=(1, 2), keepdim=True) / total
        mu_yy = (gray * dy * dy).sum(dim=(1, 2), keepdim=True) / total
        mu_xy = (gray * dx * dy).sum(dim=(1, 2), keepdim=True) / total
        trace = mu_xx + mu_yy
        det_term = torch.sqrt(((mu_xx - mu_yy) * 0.5).pow(2) + mu_xy.pow(2) + self.eps)
        lam1 = (trace * 0.5 + det_term).clamp_min(self.eps)
        lam2 = (trace * 0.5 - det_term).clamp_min(self.eps)
        axis_ratio = torch.sqrt(lam2 / lam1)
        elongation = 1.0 - axis_ratio

        flat_r = r_norm.flatten(1)
        flat_g = gray.flatten(1)
        sorted_r, order = torch.sort(flat_r, dim=1)
        sorted_g = torch.gather(flat_g, 1, order)
        cflux = sorted_g.cumsum(dim=1)
        total_flat = sorted_g.sum(dim=1, keepdim=True).clamp_min(self.eps)
        r20_idx = (cflux >= 0.2 * total_flat).float().argmax(dim=1)
        r80_idx = (cflux >= 0.8 * total_flat).float().argmax(dim=1)
        batch_idx = torch.arange(B, device=x.device)
        r20 = sorted_r[batch_idx, r20_idx].unsqueeze(1).clamp_min(self.eps)
        r80 = sorted_r[batch_idx, r80_idx].unsqueeze(1).clamp_min(self.eps)
        concentration = 5.0 * torch.log10((r80 / r20).clamp_min(1.0 + self.eps))

        gini = self._gini(gray)
        m20 = self._m20(gray, xx, yy, xc, yc)
        sersic_n, sersic_slope, sersic_err = self._sersic_proxy(gray, r_norm)

        feats = torch.cat([
            mean_int.view(B, 1), std_int.view(B, 1),
            central.view(B, 1), inner.view(B, 1), outer.view(B, 1),
            concentration.view(B, 1), asym.view(B, 1), clump.view(B, 1),
            axis_ratio.view(B, 1), elongation.view(B, 1),
            gini.view(B, 1), m20.view(B, 1),
            sersic_n, sersic_slope, sersic_err
        ], dim=1)
        return feats

class GalaxyModel(nn.Module):
    def __init__(self, num_classes=10, use_pretrained=True, dropout=0.30):
        super().__init__()
        weights = ConvNeXt_Small_Weights.IMAGENET1K_V1 if use_pretrained else None
        self.backbone = convnext_small(weights=weights)
        feat_dim = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Identity()

        self.morph = MorphologyExtractor()
        self.morph_norm = nn.LayerNorm(15)
        self.morph_mlp = nn.Sequential(
            nn.Linear(15, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 128),
            nn.GELU()
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim + 128, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = x.float()
        if x.max() > 1.5:
            x = x / 255.0

        morph_feat = self.morph_mlp(self.morph_norm(self.morph(x)))
        xn = (x - self.mean) / self.std
        img_feat = self.backbone(xn)
        fused = torch.cat([img_feat, morph_feat], dim=1)
        return self.head(fused)
"""



In [ ]:
# ── Build the improved model class from model_source ─────────
exec(model_source, globals())
print('GalaxyModel is ready.')

In [ ]:
# ── Transforms, datasets, sampler, and dataloaders ──────────
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

class RandomRotate90:
    def __call__(self, img):
        k = random.randint(0, 3)
        return transforms.functional.rotate(img, angle=90 * k)

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.88, 1.0), ratio=(0.96, 1.04)),
    RandomRotate90(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = GalaxyDataset(X_train, y_train, transform=train_transform)
val_dataset   = GalaxyDataset(X_val, y_val, transform=val_transform)

class_counts = np.bincount(y_train, minlength=10)
sample_weights = 1.0 / np.power(class_counts[y_train], 0.75)
sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples  : {len(val_dataset)}')
print('Class counts :', class_counts)



In [ ]:
# ── Initialise model, loss, optimiser ───────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

model = GalaxyModel(num_classes=10, use_pretrained=True, dropout=0.30).to(device)

class CBFocalLoss(nn.Module):
    def __init__(self, samples_per_cls, beta=0.9999, gamma=1.5):
        super().__init__()
        effective_num = 1.0 - np.power(beta, samples_per_cls)
        weights = (1.0 - beta) / np.clip(effective_num, 1e-8, None)
        weights = weights / weights.mean()
        self.register_buffer('class_weights', torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()
        idx = torch.arange(targets.size(0), device=targets.device)
        target_logp = log_probs[idx, targets]
        target_p = probs[idx, targets]
        alpha = self.class_weights.to(logits.device)[targets]
        loss = -alpha * ((1.0 - target_p).clamp(min=1e-6) ** self.gamma) * target_logp
        return loss.mean()

criterion = CBFocalLoss(np.bincount(y_train, minlength=10), beta=0.9999, gamma=1.5).to(device)

backbone_params = []
head_params = []
morph_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith('head'):
        head_params.append(param)
    elif name.startswith('morph'):
        morph_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 8.0e-5},
    {'params': morph_params,    'lr': 2.5e-4},
    {'params': head_params,     'lr': 2.5e-4},
], weight_decay=5e-4)

num_epochs = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print(f'Device        : {device}')
print(f'AMP enabled   : {use_amp}')
print(f'Parameters    : {sum(p.numel() for p in model.parameters()):,}')
print(f'Class counts  : {np.bincount(y_train, minlength=10)}')
print('Optim groups  : backbone / morphology / head')



In [ ]:
# ── Training loop (select best model by Macro-F1) ───────────
def predict_with_tta(model, images):
    logits = []
    for k in range(4):
        logits.append(model(torch.rot90(images, k=k, dims=[2, 3])))
    flipped = torch.flip(images, dims=[3])
    for k in range(4):
        logits.append(model(torch.rot90(flipped, k=k, dims=[2, 3])))
    return torch.stack(logits, dim=0).mean(dim=0)


def set_requires_grad(module, flag):
    for p in module.parameters():
        p.requires_grad = flag


def update_ema(ema_model, model, decay=0.999):
    with torch.no_grad():
        ema_state = ema_model.state_dict()
        model_state = model.state_dict()
        for k, v in ema_state.items():
            if not v.dtype.is_floating_point:
                v.copy_(model_state[k])
            else:
                v.mul_(decay).add_(model_state[k], alpha=1.0 - decay)


ema_model = copy.deepcopy(model).to(device)
for p in ema_model.parameters():
    p.requires_grad_(False)

freeze_backbone_epochs = 2
set_requires_grad(model.backbone.features, False)

best_f1 = -1.0
best_acc = -1.0
best_state = None
patience = 8
patience_counter = 0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_acc': [],
    'val_macro_f1': []
}

for epoch in range(num_epochs):
    if epoch == freeze_backbone_epochs:
        set_requires_grad(model.backbone.features, True)
        print('Backbone unfrozen for fine-tuning.')

    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for imgs, lbls in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(imgs)
            loss = criterion(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        update_ema(ema_model, model, decay=0.999)

        running_loss += loss.item() * lbls.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == lbls).sum().item()
        train_total += lbls.size(0)

    train_loss = running_loss / max(train_total, 1)
    train_acc = train_correct / max(train_total, 1)

    scheduler.step()

    ema_model.eval()
    y_true_val, y_pred_val = [], []

    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)

            outputs = predict_with_tta(ema_model, imgs)
            preds = outputs.argmax(dim=1)

            y_true_val.extend(lbls.cpu().numpy())
            y_pred_val.extend(preds.cpu().numpy())

    val_acc = accuracy_score(y_true_val, y_pred_val)
    val_macro_f1 = f1_score(y_true_val, y_pred_val, average='macro')

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_macro_f1'].append(val_macro_f1)

    current_lrs = [pg['lr'] for pg in optimizer.param_groups]
    print(
        f"Epoch [{epoch+1:02d}/{num_epochs}]  "
        f"Train Loss: {train_loss:.4f}  "
        f"Train Acc: {train_acc:.4f}  "
        f"Val Acc: {val_acc:.4f}  "
        f"Val Macro-F1: {val_macro_f1:.4f}  "
        f"LRs: {[f'{lr:.1e}' for lr in current_lrs]}"
    )

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        best_acc = val_acc
        best_state = copy.deepcopy(ema_model.state_dict())
        patience_counter = 0
        print(f"  -> Best EMA model updated (Val Macro-F1 = {best_f1:.4f}, Val Acc = {best_acc:.4f})")
    else:
        patience_counter += 1
        print(f'  -> No improvement. Early-stop counter: {patience_counter}/{patience}')

    if patience_counter >= patience:
        print('Early stopping triggered.')
        break

if best_state is None:
    raise RuntimeError('Training did not produce a valid best_state.')

model.load_state_dict(best_state)
print(f'Best validation accuracy : {best_acc*100:.2f}%')
print(f'Best validation macro-F1 : {best_f1:.4f}')

checkpoint = {
    'state_dict': model.state_dict(),
    'model_source': model_source,
    'model_class': 'GalaxyModel',
    'num_classes': 10,
}
torch.save(checkpoint, './MyImprovedModel.pth')
print('Saved checkpoint: ./MyImprovedModel.pth')



In [ ]:
# ── Save the inference preprocessing file used by run_inference.py ───────────
preprocessing_py = """
from torchvision import transforms

inference_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
"""

with open('my_preprocessing.py', 'w', encoding='utf-8') as f:
    f.write(preprocessing_py)

print('Saved preprocessing file: ./my_preprocessing.py')


In [ ]:
# ── Rebuild the class once from the saved source (sanity check) ──────────────
ns = {}
exec(model_source, ns)
ModelClass = ns['GalaxyModel']
print('Checkpoint rebuild class:', ModelClass.__name__)

### 3.1.2 Evaluation on Local Validation Set

We use precision, recall, f1-score, and support to evaluate the model on our **local validation split**.

In [ ]:
import torch
import torch.nn.functional as F

# ── Evaluate the saved model on the local validation set ─────────────────────
checkpoint = torch.load('./MyImprovedModel.pth', map_location=device)

ns = {}
exec(checkpoint['model_source'], ns)
ModelClass = ns[checkpoint['model_class']]
model_eval = ModelClass(num_classes=checkpoint['num_classes'])
model_eval.load_state_dict(checkpoint['state_dict'])
model_eval = model_eval.to(device)
model_eval.eval()

y_true = []
y_pred = []
y_scores = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = predict_with_tta(model_eval, images)
        probs = F.softmax(outputs, dim=1)
        predicted = probs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_scores.extend(probs.cpu().numpy())

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f'Validation Accuracy : {acc*100:.2f}%')
print(f'Validation Macro-F1 : {macro_f1:.4f}')

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=galaxy_classes, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Local Validation Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.plot(history['val_macro_f1'], label='Val Macro-F1')
plt.legend()
plt.title('Training Curves')
plt.xlabel('Epoch')
plt.show()

The previous plain image-only classifier is replaced by a stronger **ConvNeXt-Small + morphology-feature fusion** pipeline. In addition to deep image features, the model now extracts **Sérsic-inspired radial-profile features** together with **non-parametric morphology cues** such as concentration, asymmetry, clumpiness, axis ratio, Gini, and M20. The improved recipe combines **class-aware sampling**, **class-balanced focal loss**, **D4-consistent augmentation**, **EMA checkpointing**, and **macro-F1-based model selection** to improve both accuracy and hard-class discrimination.


## 3.2 Improvement

<div class="alert alert-warning">

### 3.2.1 Task 2

Improve the CNN model (you can also improve based on other models, but CNN is currently the most common model for image recognition)

#### Option 1: Pre-trained ResNet18

ResNet18 is a convolutional neural network architecture from the ResNet family. '18' refers to 18-layer of this architecture. 

The model has been pretrained on ImageNet (1,000 classes) and learned useful generic features for images, such as edges, textures, shapes, etc. Fine-tuning allows us to leverage these features and adapt them to our own dataset.


#### Option 2: Attention layer

Relevant papers：

[CoAtNet: Marrying Convolution and Attention for All Data Sizes](https://arxiv.org/abs/2106.04803) 

[An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929). 

The later paper intorduced ViT model which is a transformer architecture used in vision tasks. 

Mathematically, convolution of $x$ with kernel matrix $w$ is:

$$y_i=\sum_{j \in \mathcal{L}(i)} w_{i-j} \odot x_j$$

where $\mathcal{L}(i)$ denotes a local neighbors

In vision tasks:

$$\text{standard self-attention matrix}=A_{ij}=\frac{\exp \left(x_i^{\top} x_j\right)}{\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k\right)}$$


where $x_i$ represents the feature vector at spatial location $i$ in the image. 

Loosely speaking, the entries in attention matrix $A_{ij}$ means how important is position $j$ to position $i$.

Predict label $y$ using attention matrix:


$$y_i=\sum_{j \in \mathcal{G}} \frac{\exp \left(x_i^{\top} x_j\right)}{\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k\right)} x_j$$

where $\mathcal{G}$ indicates the global spatial space. 


Combine Attention and convolution (pre-normalization):

$$\text{relative self-attention}=A_{i, j}=\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k+\boldsymbol{w}_{i-k}\right)$$

So

$$y_i^{\mathrm{pre}}=\sum_{j \in \mathcal{G}} \frac{\exp \left(x_i^{\top} x_j+w_{i-j}\right)}{\sum_{k \in \mathcal{G}} \exp \left(x_i^{\top} x_k+w_{i-k}\right)} x_j$$


Further Improvment:

We can build multiple CNN layers to extract more 'meaningful' features, and use residual to prevent model deterioration. Reference paper: [ASTROFORMER: MORE DATA MIGHT NOT BE ALL YOU NEED FOR CLASSIFICATION](https://arxiv.org/abs/2304.05350)

(ASTROFORMER is the SOTA model for zoo2 galaxy classification)

#### Option 3: Others

Your own ideas are encouraged to achieve better performance.


</div>

In [ ]:
# ── 3.2.1 Improved model definition ─────────────────────────
# The improved model used in this notebook is the ConvNeXt-Small-based GalaxyModel
# with an additional handcrafted morphology branch, defined earlier from `model_source`.

model_improved = GalaxyModel(num_classes=10, use_pretrained=True, dropout=0.30).to(device)
model_improved.load_state_dict(torch.load('./MyImprovedModel.pth', map_location=device)['state_dict'])
print(model_improved)



In [ ]:
# ── 3.2.1 Improved model training ───────────────────────────
# Training was already completed in the main training section above.
# This cell keeps the notebook structure aligned with the original template.

print('Improved ConvNeXt-Small training has already been completed above.')
print('Saved file: ./MyImprovedModel.pth')


### 3.2.2 Evaluation on Local Validation Set

Please use an evaluation method similar to 3.1.2 to evaluate your model.  

As a baseline, using pre-trained ResNet18:

- **10% of whole dataset, 80/20 split** → ~65–70% accuracy  
- **30% of whole dataset, 80/20 split** → ~70% accuracy  
- **50% of whole dataset, 80/20 split** → ~75–80% accuracy  

Your solution should match or exceed these baselines for the dataset size you chose.

In [ ]:
# ── 3.2.2 Evaluate the improved model on the local validation set ────────────
model_improved.eval()

y_true_imp, y_pred_imp = [], []

with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        outputs = predict_with_tta(model_improved, imgs)
        preds = outputs.argmax(dim=1)

        y_true_imp.extend(lbls.cpu().numpy())
        y_pred_imp.extend(preds.cpu().numpy())

val_acc_imp = accuracy_score(y_true_imp, y_pred_imp)
val_macro_f1_imp = f1_score(y_true_imp, y_pred_imp, average='macro')

print(f'Improved Validation Accuracy : {val_acc_imp*100:.2f}%')
print(f'Improved Validation Macro-F1 : {val_macro_f1_imp:.4f}')
print(classification_report(y_true_imp, y_pred_imp, target_names=galaxy_classes, digits=4))

---
# 4. Final Evaluation on Held-out Test Set

Once you are happy with your model, run inference on the **fixed test set** (`X_test.npy`) provided by the instructor.

- You must **not** use `X_test.npy` in any way during training.
- Save your predictions to `y_pred_test.npy` and submit this file alongside your notebook.
- The instructor will compare your predictions against `y_test.npy` to compute your final score.

---

In [ ]:
%run run_inference.py \
    --test_h5 Galaxy10_DECals_20pct.h5 \
    --model_pth MyImprovedModel.pth \
    --transform_py my_preprocessing.py \
    --transform_name inference_transform

---
## 3.3 Classification by Human

<div class="alert alert-warning">

Look at the 25 galaxy images shown below (labels are hidden).  
Write your best guess for each galaxy class in the cell that follows.

</div>

In [ ]:
# NOTE: Read-only — do not modify

import random

random.seed(99)   # fixed seed so every student sees the same 25 images
sample_indices = [i + random.randint(0, 100) for i in range(25)]

fig = plt.figure(figsize=(20, 20))
for plot_i, img_idx in enumerate(sample_indices):
    plt.subplot(5, 5, plot_i + 1)
    plt.imshow(X_train[img_idx])
    plt.title(f'Image #{plot_i+1}', fontsize=9)
    # Labels intentionally hidden — plt.title(galaxy_classes[y_train[img_idx]])
    fig.tight_layout(pad=3.0)
plt.show()

<div class="alert alert-warning">

### ✏️ Your Human Classification Answers

Fill in your guesses below. Use the class names from `galaxy_classes`:

```
0: Disturbed Galaxies
1: Merging Galaxies
2: Round Smooth Galaxies
3: In-between Round Smooth Galaxies
4: Cigar Shaped Smooth Galaxies
5: Barred Spiral Galaxies
6: Unbarred Tight Spiral Galaxies
7: Unbarred Loose Spiral Galaxies
8: Edge-on Galaxies without Bulge
9: Edge-on Galaxies with Bulge
```

| Image # | Your Guess |
|---------|------------|
| 1  |  |
| 2  |  |
| 3  |  |
| 4  |  |
| 5  |  |
| 6  |  |
| 7  |  |
| 8  |  |
| 9  |  |
| 10 |  |
| 11 |  |
| 12 |  |
| 13 |  |
| 14 |  |
| 15 |  |
| 16 |  |
| 17 |  |
| 18 |  |
| 19 |  |
| 20 |  |
| 21 |  |
| 22 |  |
| 23 |  |
| 24 |  |
| 25 |  |

</div>